# 🕵️ Dia 36 — Mapeando um alerta publicado no ATT&CK

**Investigador Cyber 90 Dias** · Fase 3 · Rank Jutsu `B`

> Baseado no *Day 36* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-3-cyber-threat-intelligence/day-36.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

O rascunho do alerta conjunto **ALR-2026-01** (Vigia + cooperativas) chegou com 12 frases numeradas. Você começou a pintar células, inclusive as da frase 2 ("o kit *oferece* captura de código de SMS"). Rafa: *"Isso aconteceu aqui, ou é o que o fórum diz que o kit faz? Cinco cooperativas vão escrever detecção em cima da sua camada. Cada célula tem que voltar a uma frase."*

Neste notebook você vai:

1. **Jogar "mapeia ou não mapeia?"** com as 12 frases do ALR-2026-01.
2. **Montar a tabela de evidências e a camada do Navigator** por código, com validação.
3. **Calcular o `a + 2*b`** entre a sua camada e a do Rafa, sem precisar do Navigator.
4. **Repetir a comparação com o Volt Typhoon** (passo 3 do original), com plano B offline.
5. **Escrever a nota de lacunas.**

🐍 Só biblioteca padrão do Python. Nada para instalar. Funciona offline: toda célula que tenta a rede tem plano B embutido. Rode as células de cima para baixo (Shift+Enter).

## 1. As 12 frases do alerta

O alerta está em `caso-continuo/dados/dia-36/alerta-ALR-2026-01.md` no repositório. Para o notebook funcionar sem nada além dele, as frases estão embutidas abaixo, iguaizinhas ao arquivo. Tudo fictício (`.example`, IPs de documentação).

In [ ]:
# As frases do ALR-2026-01, com o identificador que o alerta usa ([S01] a [S12]).
# Um dicionário: a chave é o ID da frase; o valor é o texto literal.
FRASES = {
    "S01": 'Entre 5 e 11 de outubro de 2026, cinco cooperativas de crédito de Minas Gerais (Serra Azul, CoopVale, CredMinas, Unicoop Sul e Cooperouro) foram alvo de uma mesma campanha de phishing por SMS que se passa pela própria cooperativa.',
    "S02": 'O conjunto de ferramentas usado, apelidado internamente de IscaKit, é um kit de phishing conhecido por oferecer módulos de captura de código de SMS (OTP) e de envio automático de mensagens por WhatsApp.',
    "S03": 'Os operadores registraram domínios imitação para cada cooperativa, como serraazul-pix[.]example e coopvale-pix[.]example, entre um e dez dias antes de cada disparo.',
    "S04": 'Para cada domínio, os operadores obtiveram um certificado TLS gratuito de uma autoridade certificadora pública, em geral na véspera do disparo.',
    "S05": 'Os operadores enviaram SMS aos celulares de cooperados com o texto "sua chave Pix foi bloqueada por segurança" e um link encurtado (lnk[.]example/sa7) que redirecionava para o domínio imitação.',
    "S06": 'A página servida no domínio imitação reproduzia o internet banking da cooperativa e enviava ao servidor dos operadores o usuário e a senha digitados pelo cooperado.',
    "S07": 'Com as credenciais capturadas, os operadores entraram no internet banking legítimo da cooperativa e iniciaram transferências Pix para contas de terceiros.',
    "S08": 'Na Serra Azul, três cooperados tiveram transferências Pix não reconhecidas em 5 de outubro.',
    "S09": 'Os operadores mantiveram presença na infraestrutura ao longo da semana.',
    "S10": 'Um perfil de Instagram (@serraazul.oficial), criado pelos operadores, respondia a reclamações de cooperados com o mesmo link do SMS.',
    "S11": 'A página falsa usa um arquivo JavaScript em que aparece o comentário "// pescador", o que reforça que se trata do mesmo kit em todos os domínios.',
    "S12": 'As cooperativas devem bloquear os domínios listados no anexo e orientar os cooperados a nunca digitar a senha a partir de links recebidos por SMS.',
}
for sid, texto in FRASES.items():
    print(f"[{sid}] {texto}\n")

## 2. 🎮 Quiz: mapeia ou não mapeia?

Para cada frase, a primeira pergunta do fluxograma da CISA: **ela diz que o ator FEZ algo nesta intrusão?** Responda **S** (mapeia) ou **N** (não mapeia). Depois o notebook mostra a decisão do Rafa e o porquê.

Lembre das armadilhas: capacidade não é observação; persistência sem mecanismo para na tática; frases com dois comportamentos viram duas linhas; artefato do kit não é comportamento.

In [ ]:
# Mapeamento de referência do Rafa (o "outro analista" do passo de comparação da CISA).
# Para cada frase: (mapeia?, lista de (tática, técnica), motivo)
# Técnica None = parou na tática, porque o texto não nomeia o mecanismo.
# IDs conferidos no Enterprise ATT&CK v19.2 (e Mobile v19.2 para o T1660) em 2026-10-12.
RAFA = {
    "S01": (False, [], "Resumo: repete o que S05 detalha. Mapear duas vezes infla a camada."),
    "S02": (False, [], "CAPACIDADE ('é conhecido por oferecer'). Ninguém viu o kit pedir OTP neste caso."),
    "S03": (True, [("Resource Development", "T1583.001")], "Registrou domínios imitação: T1583.001 Domains."),
    "S04": (True, [("Resource Development", "T1588.004")],
            "OBTEVE certificado de uma CA pública: T1588.004. O texto não diz como instalou: nada de T1608.003."),
    "S05": (True, [("Initial Access", "T1566.002"), ("Resource Development", "T1608.005"),
                   ("Initial Access (Mobile)", "T1660")],
            "Dois comportamentos: o SMS com link (Enterprise T1566.002; no celular, Mobile T1660) "
            "e o encurtador preparado para redirecionar (T1608.005 Link Target)."),
    "S06": (True, [("Credential Access", "T1056.003"), ("Stealth", "T1684.001")],
            "Dois comportamentos: imitar o internet banking (T1684.001 Impersonation, que substituiu o T1656) "
            "e capturar a senha digitada (T1056.003 Web Portal Capture)."),
    "S07": (True, [("Initial Access", "T1078"), ("Impact", "T1657")],
            "Dois comportamentos: entrar com credenciais válidas (T1078) e fazer o Pix (T1657 Financial Theft)."),
    "S08": (False, [], "Consequência para as vítimas, não uma ação nova do ator (o roubo já está em S07)."),
    "S09": (True, [("Persistence", None)], "Persistência sem mecanismo nomeado: SÓ a tática. Não invente técnica."),
    "S10": (True, [("Resource Development", "T1585.001"), ("Initial Access", "T1566.003")],
            "Dois comportamentos: criar o perfil (T1585.001 Social Media Accounts) e mandar o link "
            "por uma rede social (T1566.003 Spearphishing via Service)."),
    "S11": (False, [], "ARTEFATO do kit ('// pescador'): ótimo para rastrear (Dia 38), mas não é técnica a mapear."),
    "S12": (False, [], "Recomendação da Vigia, não ação do ator."),
}

MODO_JOGO = True   # False = só mostra a decisão do Rafa

acertos = 0
for sid, texto in FRASES.items():
    mapeia, itens, motivo = RAFA[sid]
    print(f"\n[{sid}] {texto}")
    if MODO_JOGO:
        r = ""
        while r not in ("S", "N"):
            r = input("   Mapeia? (S/N): ").strip().upper()
        ok = (r == "S") == mapeia
        acertos += ok
        print("   ✅" if ok else "   ❌", "Rafa:", "MAPEIA" if mapeia else "NÃO MAPEIA")
    print("   →", motivo)
    for tatica, tecnica in itens:
        print(f"      {tatica}: {tecnica or '(só a tática)'}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(FRASES)}")

## 3. Sua tabela de evidências e sua camada

A célula abaixo traz **o seu rascunho da abertura** (o que você pintou antes do Rafa intervir). Ele tem erros de propósito. Edite o dicionário `MEU_MAPA` (apague, troque, acrescente linhas) até ficar com o que você defende, e rode.

Regras que a célula aplica por você:
- toda técnica precisa de uma frase (`S##`) e de uma citação; sem isso, sai da camada;
- o ID precisa ter o formato `T####` ou `T####.###`;
- o T1660 (Mobile) não pode entrar numa camada `enterprise-attack`: vai para uma lista à parte.

In [ ]:
import json   # para escrever e validar a camada
import re     # para validar o formato dos IDs

# Seu rascunho: técnica -> frase de evidência. EDITE à vontade.
MEU_MAPA = {
    "T1583.001": "S03",
    "T1588.004": "S04",
    "T1608.003": "S04",   # rascunho: "instalou o certificado"... o texto diz isso?
    "T1566.002": "S05",
    "T1660":     "S05",   # Mobile!
    "T1056.003": "S06",
    "T1078":     "S07",
    "T1657":     "S07",
    "T1111":     "S02",   # rascunho: interceptação de MFA... era capacidade?
    "T1547.001": "S09",   # rascunho: Registry Run Keys... onde está escrito?
}
MOBILE = {"T1660"}                                   # IDs que moram na matriz Mobile
FORMATO = re.compile(r"^T\d{4}(\.\d{3})?$")       # T#### ou T####.###

tabela, camada_tecnicas, so_mobile = [], [], []
for tid, sid in sorted(MEU_MAPA.items()):
    if not FORMATO.match(tid) or sid not in FRASES:   # sem formato ou sem frase = fora
        print(f"⛔ {tid}: formato inválido ou frase inexistente ({sid}).")
        continue
    citacao = FRASES[sid]
    tabela.append((sid, citacao, tid))
    if tid in MOBILE:
        so_mobile.append(tid)                       # registro à parte, não entra na camada Enterprise
        continue
    camada_tecnicas.append({"techniqueID": tid, "score": 1,
                            "comment": f"{sid}: {citacao[:120]}", "showSubtechniques": True})

minha_camada = {
    "name": "IscaKit, meu mapeamento do ALR-2026-01",
    "versions": {"layer": "4.5", "attack": "19", "navigator": "5.3.2"},
    "domain": "enterprise-attack",
    "description": "Mapeado do ALR-2026-01 (2026-10-12, fictício). Uma técnica por linha de evidência. ATT&CK v19.2.",
    "techniques": camada_tecnicas,
    "gradient": {"colors": ["#ffffff", "#66b1ff"], "minValue": 0, "maxValue": 1},
}
texto_json = json.dumps(minha_camada, ensure_ascii=False, indent=1)
json.loads(texto_json)                               # a "vírgula sobrando" do original: se falhar, o JSON é inválido
with open("minha-camada-alr-2026-01.json", "w", encoding="utf-8") as f:
    f.write(texto_json)

print("| Frase | Citação | Técnica |\n|---|---|---|")
for sid, cit, tid in tabela:
    print(f"| {sid} | {cit[:70]}… | {tid} |")
print(f"\n💾 minha-camada-alr-2026-01.json: {len(camada_tecnicas)} técnicas Enterprise.")
print("📱 Registradas à parte (Mobile):", so_mobile or "nenhuma")

## 4. A comparação `a + 2*b`, em Python

No Navigator você faria "Create Layer from other layers" com `a + 2*b`. A conta é simples: para cada técnica, `a` vale 1 se está na sua camada, `b` vale 1 se está na do Rafa.

- **1** = só você → reconfira sua evidência;
- **2** = só o Rafa → você deixou passar (ou discorda, com motivo);
- **3** = os dois.

In [ ]:
def comparar(a_ids, b_ids):
    """Devolve {técnica: pontuação} com a regra a + 2*b do original."""
    todas = set(a_ids) | set(b_ids)                       # união: tudo que aparece em alguma camada
    return {t: (t in a_ids) * 1 + (t in b_ids) * 2 for t in sorted(todas)}

a = {t["techniqueID"] for t in camada_tecnicas}             # sua camada (só Enterprise)
b = {tec for (_, itens, _) in RAFA.values() for (_, tec) in itens
     if tec and tec not in MOBILE}                          # camada do Rafa (só Enterprise)

LEGENDA = {1: "só você", 2: "só o Rafa", 3: "os dois"}
resultado = comparar(a, b)
for pontos in (1, 2, 3):
    lista = [t for t, p in resultado.items() if p == pontos]
    print(f"{pontos} = {LEGENDA[pontos]:<10} ({len(lista)}): {', '.join(lista) or '—'}")

# Camada de comparação pronta para o Navigator, com gradiente de 3 cores e legenda (passo 4)
comparacao = {
    "name": "ALR-2026-01: eu (a) x Rafa (b), a + 2*b",
    "versions": {"layer": "4.5", "attack": "19", "navigator": "5.3.2"},
    "domain": "enterprise-attack",
    "techniques": [{"techniqueID": t, "score": p} for t, p in resultado.items()],
    "gradient": {"colors": ["#fff2cc", "#dae8fc", "#d5e8d4"], "minValue": 1, "maxValue": 3},
    "legendItems": [{"label": "1 = só eu", "color": "#fff2cc"},
                    {"label": "2 = só o Rafa", "color": "#dae8fc"},
                    {"label": "3 = os dois", "color": "#d5e8d4"}],
}
with open("comparacao-alr-2026-01.json", "w", encoding="utf-8") as f:
    json.dump(comparacao, f, ensure_ascii=False, indent=1)
print("\n💾 comparacao-alr-2026-01.json salvo.")
print("A tática Persistence (S09) não vira célula: camada do Navigator é por técnica. Registre na nota.")

## 5. Volt Typhoon: a sua amostra × o MITRE (passos 3 e 4 do original)

O passo 3 baixa a camada oficial do grupo G1017. A célula tenta a rede; se falhar, usa uma lista embutida com as técnicas que têm relacionamento `uses` **direto** do G1017 no dataset Enterprise v19.2 (81 técnicas, extraídas em 2026-10-12).

⚠️ Atenção ao que o Dia 38 vai explicar: a camada oficial **também** traz técnicas de campanhas atribuídas ao grupo, então ela é maior que a lista direta (o original contou 98 com `score==1` em setembro de 2026). Anote qual das duas você usou.

Sua camada `a` aqui são as cinco linhas resolvidas do passo 1 do original (seis técnicas).

In [ ]:
import urllib.request   # para tentar baixar a camada oficial

VOLT_DIRETO_V19_2 = ['T1003.001', 'T1003.003', 'T1005', 'T1006', 'T1007', 'T1010', 'T1012', 'T1016', 'T1016.001', 'T1018', 'T1021.001', 'T1027.002', 'T1033', 'T1036.005', 'T1036.008', 'T1046', 'T1047', 'T1049', 'T1056.001', 'T1057', 'T1059.001', 'T1059.003', 'T1059.004', 'T1068', 'T1069', 'T1069.001', 'T1069.002', 'T1070.004', 'T1070.007', 'T1074', 'T1074.001', 'T1078', 'T1078.002', 'T1083', 'T1087.001', 'T1087.002', 'T1090', 'T1090.001', 'T1090.003', 'T1105', 'T1112', 'T1113', 'T1120', 'T1124', 'T1133', 'T1140', 'T1190', 'T1217', 'T1218', 'T1497.001', 'T1505.003', 'T1518', 'T1552', 'T1552.004', 'T1555', 'T1555.003', 'T1560.001', 'T1570', 'T1573.001', 'T1584.003', 'T1584.004', 'T1584.005', 'T1584.008', 'T1587.004', 'T1588.002', 'T1588.006', 'T1589', 'T1589.002', 'T1590', 'T1590.004', 'T1590.006', 'T1591', 'T1591.004', 'T1592', 'T1593', 'T1594', 'T1596.005', 'T1614', 'T1654', 'T1680', 'T1685.005']

URL_G1017 = "https://attack.mitre.org/groups/G1017/G1017-enterprise-layer.json"
try:
    with urllib.request.urlopen(URL_G1017, timeout=20) as r:
        oficial = json.load(r)
    b_volt = {t["techniqueID"] for t in oficial["techniques"] if t.get("score") == 1}
    origem_b = f"camada oficial G1017 (rede), {len(b_volt)} técnicas com score 1"
except Exception as erro:
    b_volt = set(VOLT_DIRETO_V19_2)
    origem_b = f"plano B: relacionamentos diretos do G1017 no v19.2 ({len(b_volt)} técnicas) — {erro.__class__.__name__}"

# As cinco linhas resolvidas do original (a linha 3 virou duas técnicas)
a_volt = {"T1003.003", "T1090.001", "T1090.003", "T1584.008", "T1190", "T1654"}

res_volt = comparar(a_volt, b_volt)
print("Origem de b:", origem_b)
for pontos in (1, 3):
    print(f"{pontos} = {LEGENDA[pontos].replace('o Rafa', 'o MITRE')}:",
          [t for t, p in res_volt.items() if p == pontos])
print("2 = só o MITRE:", sum(1 for p in res_volt.values() if p == 2), "técnicas (a maioria vem de outras fontes)")

## 6. ✍️ Nota de lacunas (passo 5)

Três partes, meia página no máximo. A célula monta o esqueleto a partir dos números acima; você completa o raciocínio.

In [ ]:
#@title Nota de lacunas { display-mode: "form" }
so_eu_ainda_sustento = "T1608.003 cai: S04 diz que obteve o certificado, não que instalou. T1111 cai: S02 é capacidade. T1547.001 cai: S09 não nomeia mecanismo."  #@param {type:"string"}
so_rafa_por_que_perdi = "Não dividi S05, S06 e S10 em dois comportamentos cada; e esqueci o perfil de Instagram (S10)."  #@param {type:"string"}
camada_de_grupo = "A camada do G1017 junta muitos relatórios e campanhas; ela não é o mapeamento de um alerta só."  #@param {type:"string"}

so_eu = [t for t, p in resultado.items() if p == 1]
so_b = [t for t, p in resultado.items() if p == 2]
nota = f"""# Nota de lacunas — ALR-2026-01 (ATT&CK v19.2)

## Só eu mapeei (pontuação 1): {', '.join(so_eu) or '—'}
{so_eu_ainda_sustento}

## Só o Rafa mapeou (pontuação 2): {', '.join(so_b) or '—'}
{so_rafa_por_que_perdi}

## Tática sem técnica
S09 → Persistence (o texto não nomeia mecanismo).

## Registro Mobile
{', '.join(so_mobile) or 'nenhum'} (disparo do SMS, matriz Mobile v19.2).

## Volt Typhoon (passo 5 do original)
Comparação contra: {origem_b}. Só o MITRE: {sum(1 for p in res_volt.values() if p == 2)} técnicas.
{camada_de_grupo}
"""
with open("nota-de-lacunas-dia36.md", "w", encoding="utf-8") as f:
    f.write(nota)
print(nota)

try:
    from google.colab import files
    for nome in ["minha-camada-alr-2026-01.json", "comparacao-alr-2026-01.json", "nota-de-lacunas-dia36.md"]:
        files.download(nome)
except ImportError:
    print("(Fora do Colab: os arquivos ficaram na pasta atual.)")

## ✅ Antes de fechar o Dia 36

- [ ] Quiz "mapeia ou não mapeia?" feito, e você sabe por que S02, S08, S11 e S12 não geram técnica
- [ ] `minha-camada-alr-2026-01.json` com comentário (frase) em toda técnica
- [ ] `comparacao-alr-2026-01.json` carregado no Navigator com a legenda 1/2/3
- [ ] S09 parou na tática
- [ ] Nota de lacunas escrita

**Amanhã (Dia 37):** o que entra num perfil de ator, e por que o perfil do "Pescador" vai ser o perfil de um **cluster de atividade**, nunca de uma pessoa.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>